# SCE detector variation — selection comparison (Sept 2026)

Investigate whether **0xSCE**, **2xSCE**, and **CV** change the events that pass the `sel_mup` selection.

**Workflow**
1. Find truth-level events common to all three samples (match on `run`, `subrun`, `evt`, and neutrino energy `E` from `meta`).
2. Among those common events, determine which appear in each variation's `evt` dataframe (selected events).
3. Plot neutrino energy and final-selected kinematic variables for the three variations.
4. Print and save events **exclusively** selected in one variation (selected there but not in the other two), with percentages.

Heavy file I/O is cached under `OUT_DIR/cache/`. Processing is **sequential** (one file at a time) with a memory guard (`MAX_MEM_FRACTION`, default 60%) — no multiprocessing pool that duplicates the multi-million-entry common-key set.

In [ ]:
import sys
from os import makedirs, path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana')
plt.style.use('presentation.mplstyle')

from analysis_village.numucc_1p0pi.scripts.sce_selection_comparison import (
    DEFAULT_BASE,
    DEFAULT_OUT,
    DEFAULT_VARIATIONS,
    keys_to_dataframe,
    run_analysis,
)

In [ ]:
# ── Input / output paths ─────────────────────────────────────────────────────
BASE_DIR = DEFAULT_BASE
VARIATIONS = DEFAULT_VARIATIONS
OUT_DIR = DEFAULT_OUT
FIG_DIR = path.join(OUT_DIR, 'plots')
EXCL_DIR = path.join(OUT_DIR, 'exclusive_events')
makedirs(FIG_DIR, exist_ok=True)

# Set True to force a full rescan (slow: ~3000 files × 3 variations, sequential)
RERUN_META = False
RERUN_SELECTION = False
RERUN_HISTS = False
MAX_MEM_FRACTION = 0.70  # abort if system memory exceeds this fraction

print('Variations:')
for name, sub in VARIATIONS.items():
    print(f'  {name}: {path.join(BASE_DIR, sub)}')
print(f'Output: {OUT_DIR}')

In [ ]:
result = run_analysis(
    base_dir=BASE_DIR,
    variations=VARIATIONS,
    out_dir=OUT_DIR,
    rerun_meta=RERUN_META,
    rerun_selection=RERUN_SELECTION,
    rerun_hists=RERUN_HISTS,
    max_used_fraction=MAX_MEM_FRACTION,
)

n_common = result['n_common']
selected = result['selected']
exclusive = result['exclusive']
summary = result['summary']
hist_payload = result['hist_payload']

print(f'Common truth events (all three): {n_common:,}')

## Selection summary

Counts are restricted to the **common truth-event pool**. Exclusive events are selected in one variation but **not** in the other two.

In [ ]:
summary_display = summary.copy()
for col in ['frac_common_selected', 'frac_common_exclusive', 'frac_selected_exclusive']:
    summary_display[col] = (100 * summary_display[col]).map(lambda x: f'{x:.3f}%' if pd.notna(x) else '')
display(summary_display)

print('\nPairwise symmetric differences among selected common events:')
names = list(VARIATIONS.keys())
for i, a in enumerate(names):
    for b in names[i + 1:]:
        symdiff = selected[a] ^ selected[b]
        print(f'  {a} vs {b}: {len(symdiff):,} ({100*len(symdiff)/n_common:.3f}% of common)')

## Exclusive selections

Events in the common pool that pass selection in exactly one variation. CSVs are written to `exclusive_events/`.

In [ ]:
for name, keys in exclusive.items():
    n_excl = len(keys)
    n_sel = len(selected[name])
    pct_common = 100 * n_excl / n_common
    pct_sel = 100 * n_excl / n_sel if n_sel else 0.0
    print(f'\n=== Exclusive to {name} ===')
    print(f'  count: {n_excl:,}')
    print(f'  {pct_common:.3f}% of common truth events')
    print(f'  {pct_sel:.3f}% of {name} selected events')
    df_excl = keys_to_dataframe(keys)
    out_csv = path.join(EXCL_DIR, f'exclusive_{name}.csv')
    print(f'  saved: {out_csv}')
    if n_excl:
        display(df_excl.head(20))
    else:
        print('  (none)')

## Variable distributions (common-truth selected events)

Normalized distributions for events that pass `sel_mup` in each variation, restricted to the common truth pool. Legend includes total selected count.

In [ ]:
VAR_COLORS = {'CV': 'C2', '0xSCE': 'C0', '2xSCE': 'C1'}
PLOT_VAR_DEFS = hist_payload['var_defs']
ALL_HISTS = hist_payload['hists']
N_EVTS = hist_payload['n_evts']

PRIORITY_VARS = [
    'E_nu', 'muon-p', 'proton-p', 'tki-del_Tp', 'tki-del_alpha',
    'vertex_z', 'muon-dir_z', 'proton-dir_z',
]
PLOT_VARS = [v for v in PRIORITY_VARS if v in PLOT_VAR_DEFS]
PLOT_VARS += [v for v in PLOT_VAR_DEFS if v not in PLOT_VARS]


def save_fig(fig, name, dpi=150):
    for ext in ('pdf', 'png'):
        fig.savefig(path.join(FIG_DIR, f'{name}.{ext}'), bbox_inches='tight', dpi=dpi)
    print(f'  saved {name}.pdf / .png')


def plot_var_comparison(var_name, var_cfg, reference='CV', ratio=True):
    bins = np.asarray(var_cfg['bins'])
    fig, axes = plt.subplots(2 if ratio else 1, 1, figsize=(7, 6),
                             gridspec_kw={'height_ratios': [3, 1]} if ratio else None,
                             sharex=True)
    ax_top = axes[0] if ratio else axes
    ax_bot = axes[1] if ratio else None

    ref_counts = ALL_HISTS[reference][var_name].copy()
    ref_total = ref_counts.sum()
    if ref_total > 0:
        ref_norm = ref_counts / ref_total
    else:
        ref_norm = None

    for var_label in VARIATIONS:
        counts = ALL_HISTS[var_label][var_name].copy()
        total = counts.sum()
        if total == 0:
            continue
        norm = counts / total
        label = f"{var_label} (N={N_EVTS[var_label]:,})"
        ax_top.step(bins, np.append(norm, norm[-1]), where='post',
                    label=label, color=VAR_COLORS.get(var_label), linewidth=1.5)

    ax_top.set_ylabel('Events (normalised)')
    ax_top.legend(fontsize=8, frameon=True)
    ax_top.set_title(var_cfg['label'], fontsize=10)

    if ratio and ax_bot is not None and ref_norm is not None:
        ax_bot.axhline(1.0, color='k', linewidth=0.8, linestyle='--')
        for var_label in VARIATIONS:
            if var_label == reference:
                continue
            counts = ALL_HISTS[var_label][var_name].copy()
            total = counts.sum()
            if total == 0:
                continue
            norm = counts / total
            with np.errstate(divide='ignore', invalid='ignore'):
                rat = np.where(ref_norm > 0, norm / ref_norm, np.nan)
            ax_bot.step(bins, np.append(rat, rat[-1]), where='post',
                        color=VAR_COLORS.get(var_label), linewidth=1.5)
        ax_bot.set_ylabel(f'/ {reference}')
        ax_bot.set_ylim(0.8, 1.2)
        ax_bot.set_xlabel(var_cfg['label'])

    fig.tight_layout()
    return fig

print('Selected event counts (common-truth pool):')
for name, n in N_EVTS.items():
    print(f'  {name}: {n:,}')

In [ ]:
for var_name in PLOT_VARS:
    var_cfg = PLOT_VAR_DEFS[var_name]
    fig = plot_var_comparison(var_name, var_cfg, reference='CV')
    save_fig(fig, f'selection_{var_name}')
    plt.show()
    plt.close(fig)

## Answer

Use the summary table and exclusive-event counts above to answer whether SCE shifts the selected sample. A ~5% change in total selected events would correspond to roughly 5% of the common-truth pool differing between variations; check the pairwise symmetric differences and exclusive-event percentages.